# Car Mileage Prediction Using Multiple Linear Regression

**CIE-2 Machine Learning Activity – Activity 1**

## Problem statement
Predict a car's fuel efficiency (`mpg`, miles per gallon) using multiple vehicle features from the UCI Auto MPG dataset.

## Objectives
1. Load and understand the Auto MPG dataset.
2. Clean missing/invalid values and prepare features.
3. Train a Single Linear Regression baseline using `weight`.
4. Train a Multiple Linear Regression model using several features.
5. Compare both models using R², Adjusted R², and RMSE.
6. Explain one important modelling decision.

**Dataset source:** https://archive.ics.uci.edu/dataset/9/auto+mpg  
**Raw data file:** https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data


## 1. Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option("display.max_columns", None)


## 2. Load the dataset

In [ ]:
# The original UCI data file is whitespace-separated and has no header row.
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"

columns = [
    "mpg", "cylinders", "displacement", "horsepower",
    "weight", "acceleration", "model_year", "origin", "car_name"
]

df = pd.read_csv(
    url,
    sep=r"\s+",
    names=columns,
    na_values="?",
    comment="\t",
    skipinitialspace=True
)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Explore and clean the data

In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values before cleaning:")
display(df.isnull().sum())

# Horsepower contains '?' values in the original file; these are read as NaN.
df = df.dropna().copy()

# car_name is an identifier/text field, not used as a numeric predictor.
print("\nDataset shape after removing missing rows:", df.shape)
display(df.describe())


## 4. Select input features and target

In [ ]:
# Target: fuel efficiency in miles per gallon
target = "mpg"

# Multiple input features
features = [
    "cylinders", "displacement", "horsepower",
    "weight", "acceleration", "model_year", "origin"
]

X = df[features].copy()
y = df[target].copy()

print("Input features:", features)
print("Target:", target)


## 5. Exploratory data analysis

In [ ]:
plt.figure(figsize=(9, 6))
sns.heatmap(df[features + [target]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap – Auto MPG")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="weight", y="mpg")
plt.title("Car Weight vs Mileage")
plt.xlabel("Weight")
plt.ylabel("Mileage (MPG)")
plt.tight_layout()
plt.show()


## 6. Split data into training and testing sets

In [ ]:
# Use the same split for both models to make the comparison fair.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 7. Train Single Linear Regression (baseline)

In [ ]:
# Baseline model uses only one feature: weight.
single_feature = ["weight"]

single_model = LinearRegression()
single_model.fit(X_train[single_feature], y_train)

single_predictions = single_model.predict(X_test[single_feature])

print("Single Linear Regression coefficient:", single_model.coef_[0])
print("Single Linear Regression intercept:", single_model.intercept_)


## 8. Train Multiple Linear Regression

In [ ]:
multiple_model = LinearRegression()
multiple_model.fit(X_train, y_train)

multiple_predictions = multiple_model.predict(X_test)

coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": multiple_model.coef_
})
display(coefficients)

print("Intercept:", multiple_model.intercept_)


## 9. Evaluate and compare both models

In [ ]:
def regression_metrics(y_true, y_pred, number_of_features):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    n = len(y_true)
    p = number_of_features

    # Adjusted R-squared; defined when n > p + 1.
    adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    return {
        "R-squared": r2,
        "Adjusted R-squared": adjusted_r2,
        "RMSE": rmse
    }

results = pd.DataFrame([
    {
        "Model": "Single Linear Regression (weight only)",
        **regression_metrics(y_test, single_predictions, 1)
    },
    {
        "Model": "Multiple Linear Regression (7 features)",
        **regression_metrics(y_test, multiple_predictions, len(features))
    }
])

display(results.round(4))

best_model_name = results.loc[results["RMSE"].idxmin(), "Model"]
print("Model with lower test RMSE:", best_model_name)


## 10. Actual vs predicted values

In [ ]:
comparison = pd.DataFrame({
    "Actual MPG": y_test.values,
    "Predicted MPG": multiple_predictions
})
display(comparison.head(10))

plt.figure(figsize=(7, 5))
plt.scatter(y_test, multiple_predictions, alpha=0.7)
low = min(y_test.min(), multiple_predictions.min())
high = max(y_test.max(), multiple_predictions.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual MPG")
plt.ylabel("Predicted MPG")
plt.title("Actual vs Predicted Mileage – Multiple Linear Regression")
plt.tight_layout()
plt.show()


## 11. Predict mileage for a new car

Enter example vehicle details below. These are illustrative inputs; change them to test other cars.

In [ ]:
new_car = pd.DataFrame([{
    "cylinders": 4,
    "displacement": 140,
    "horsepower": 90,
    "weight": 2500,
    "acceleration": 15.5,
    "model_year": 82,
    "origin": 1
}])

predicted_mpg = multiple_model.predict(new_car[features])[0]
print(f"Predicted mileage: {predicted_mpg:.2f} MPG")


## 12. Conclusion

The project trains and compares a one-feature Linear Regression baseline with a Multiple Linear Regression model using seven vehicle features. Use the printed test-set R², Adjusted R², and RMSE values to report which model performed better. A lower RMSE indicates smaller prediction errors, while higher R² indicates that the model explains more variation in mileage.

## 13. Decision log

**Decision:** Remove rows where horsepower is missing (`?`) after converting these values to missing data.

**Why:** Linear Regression requires numeric input values. Removing a small number of incomplete rows provides a straightforward, reproducible preprocessing step for this beginner project.

**Additional note:** Vehicle features such as cylinders, displacement, horsepower, and weight may be correlated. Review the correlation heatmap and interpret coefficients carefully; correlation between predictors can make individual coefficients less stable.

## 14. References
- UCI Machine Learning Repository – Auto MPG: https://archive.ics.uci.edu/dataset/9/auto+mpg
- Scikit-learn LinearRegression: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html
- Scikit-learn model evaluation: https://scikit-learn.org/stable/modules/model_evaluation.html
